# Meeskond Operatsioonid - Week 8 API pipeline UrbanStyle.ltd jaoks

**Nädal 8: Python API ja automatiseeritud pipeline**  
**Meeskonnatöö: API päringud, puhastamine, RFM + marketing analytics, eksport ja automatiseerimine**

Notebook koondab Week 8 töö samas stiilis nagu Week 7 RFM notebook: iga roll on eraldi plokina, kood on käivitatav ning lõpus on Markole mõeldud äriline kokkuvõte. Week 8 edasiarendus muudab Week 7 analüüsi modulaarseks pipeline'iks, mis oskab andmeid API kaudu küsida, tulemusi valideerida, eksportida ja vajadusel teavitusi saata.

**Rollid**
- Roll A: Data Fetching - Supabase API, pagination, retry ja fallback-andmed.
- Roll B: Transform - puhastamine, ühendamine, KPI-d, RFM, CLV, cohort retention ja kampaaniaplaan.
- Roll C: Visualization + Saving - juhendi järgi KPI kokkuvõte, nädalase tulu graafik ning CSV/HTML eksport.
- Roll D: Automation Script - kogu protsessi orkestreerimine, valideerimine, logimine ja teavitused.

## Roll A: Data Fetching - API päringud ja fallback

Roll A eesmärk on tuua müügi-, kliendi- ja tooteandmed Supabase API-st. Pipeline sisaldab pagination'i, retry loogikat ning fallback'i, et analüüs oleks käivitatav ka siis, kui API võtmeid pole või ühendus ebaõnnestub.

Selles notebookis on vaikimisi kasutusel live API režiim (`USE_LIVE_API = True`). Kui Supabase võtmeid pole või ühendus ebaõnnestub, kasutab pipeline fallback/sample andmeid, et notebook jääks siiski käivitatavaks.

In [ ]:
# 1. TEEGID
# Impordime põhiteegid failiteede, ajatemplite ja tabelitöötluse jaoks.
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

# Kuvaseaded: näitame notebookis rohkem veerge ja hoiame tabelid loetavamad.
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

# 2. PROJEKTI JA MOODULITE ASUKOHT
# Leiame projekti juurkausta, et notebook töötaks nii repo juurest kui week-08/team kaustast käivitades.
ROOT = Path.cwd()
if not (ROOT / 'week-08').exists():
    ROOT = Path.cwd().parents[1]

# Week 8 tiimitöö .py moodulid asuvad selles kaustas.
TEAM_DIR = ROOT / 'week-08' / 'team'

# Lisame tiimikausta import path'i, et Python leiaks data_fetcher.py, pipeline.py, transform.py ja visualize_export.py.
sys.path.insert(0, str(TEAM_DIR))

# 3. WEEK 8 MOODULITE IMPORT
# data_fetcher: fallback/sample andmed juhuks, kui live API ei tööta.
from data_fetcher import csv_fallback_data, sample_data

# pipeline: config, live API extract, transformatsioon, valideerimine ja teavituse kokkuvõte.
from pipeline import apply_cli_date, extract, load_config, notification_summary, transform_data, validate_results

# transform: eraldi näitamiseks, kuidas andmed liidetakse ja puhastatakse.
from transform import merge_datasets, clean_data

# visualize_export: kasutame ainult juhendis nõutud KPI kokkuvõtet ja nädalase tulu graafikut.
from visualize_export import create_kpi_summary, create_weekly_chart

# 4. CONFIG JA LIVE API VALIK
# Laeme config.yaml seaded ning määrame analüüsi lõppkuupäevaks 2025-02-28.
config = apply_cli_date(load_config(), '2025-02-28')

# True = notebook proovib esmalt Supabase API-st pärida.
# Kui ühendus ebaõnnestub, saab allpool kasutada fallback/sample andmeid.
USE_LIVE_API = True

# 5. ANDMETE LAADIMINE
# Kui live API on lubatud, käivitub Roll A extract(config): sales, customers ja products tabelite päring.
if USE_LIVE_API:
    sales, customers, products, data_source = extract(config)
else:
    # Kui live API ei ole lubatud, proovime esmalt kohalikke CSV fallback-andmeid.
    fallback = csv_fallback_data()
    if fallback is not None:
        sales, customers, products = fallback
        data_source = 'csv_fallback'
    else:
        # Viimase varuna kasutame väikest näidisandmestikku, et notebook jääks käivitatavaks.
        sales, customers, products = sample_data()
        data_source = 'sample_data'

# 6. ESIMENE KONTROLL
# Prindime laaditud tabelite suurused ja kuvame sales tabeli esimesed read.
print('Andmeallikas:', data_source)
print('Sales shape:', sales.shape)
print('Customers shape:', customers.shape)
print('Products shape:', products.shape)
sales.head()

## Roll B: Transform - puhastamine, KPI-d ja RFM

Roll B ühendab müügi-, kliendi- ja tooteandmed, eemaldab vigased read ning arvutab juhendis nõutud nädalased koondnäitajad ja KPI-d. Lisaks on meeskonna töös säilinud RFM segmentide ja kampaaniaplaani tabelid, sest need toetavad Markole antavat ärilist soovitust.

In [ ]:
# Ühendame müügi-, kliendi- ja tooteandmed üheks analüüsitabeliks.
merged = merge_datasets(sales, customers, products)

# Puhastame duplikaadid, vigased kuupäevad, puuduva customer_id ja mittepositiivse müügisumma.
clean = clean_data(merged)

print('Ühendatud shape:', merged.shape)
print('Puhastatud shape:', clean.shape)
clean.head()

In [ ]:
# Käivitame kogu transformatsiooni: puhastus, KPI-d, RFM, cohort retention ja kampaaniaplaan.
results = transform_data(sales, customers, products, config, data_source=data_source)

# Kuvame juhtimiseks kõige olulisemad koondnäitajad.
print('KPI-d:')
display(pd.DataFrame([results['kpis']]))

# Andmekvaliteedi tabel näitab, mitu rida puhastuse käigus eemaldati ja miks.
print('Andmekvaliteet:')
display(results['data_quality'])

# Segmentide kokkuvõte näitab klientide arvu, käivet ja kontaktitavust segmendi kaupa.
print('RFM segmentide kokkuvõte:')
display(results['segment_summary'])

## Roll C: Visualization + Saving - diagrammid ja ekspordid

Roll C loob juhendis nõutud Plotly väljundid: nädalase tulu joondiagrammi ja KPI kokkuvõtte. Lisaks salvestatakse CSV failid ajatempliga failinimedega. Notebook ei loo juurde eemaldatud lisagraafikuid.

In [ ]:
# KPI tabel annab Markole kiire ülevaate kogukäibest, tellimustest ja keskmisest ostukorvist.
create_kpi_summary(results['kpis']).show()

# Nädalase tulu joonis on juhendi Roll C põhinõue: Marko näeb tululiikumist nädalate lõikes.
create_weekly_chart(results['weekly']).show()

In [ ]:
# Lisagraafikuid siin ei loo, sest Week 8 juhendi järgi piisab weekly chart'ist ja KPI kokkuvõttest.
# Kuvame täiendavad tulemused tabelitena, et analüüs oleks kontrollitav ilma kustutatud graafikuid tagasi lisamata.
print('Nädalased koondnäitajad:')
display(results['weekly'].head())

print('RFM segmentide tabel:')
display(results['segment_summary'])

# Top 10 klientide tabel aitab Markol näha suurima kogukulutusega kliente ilma uut graafikut lisamata.
top_10_clients = results['rfm'].nlargest(10, 'monetary_value')[
    ['customer_id', 'customer_name', 'Segment', 'frequency', 'monetary_value', 'avg_order_value', 'estimated_clv_6m']
]

print('Top 10 klienti kogukulutuse järgi:')
display(top_10_clients)

In [ ]:
# Kampaaniaplaan seob iga RFM segmendi konkreetse sõnumi, kanali, pakkumise ja mõõdikuga.
# Kuvame selle DataFrame'ina, mitte uue graafikuna.
print('Kampaaniaplaan:')
display(results['campaign_plan'])

# A/B testiplaan lisab kontrollgrupi, et kampaaniate mõju oleks mõõdetav.
print('A/B testiplaan:')
display(results['ab_test_plan'])

## Roll D: Automation Script - valideerimine, eksport ja teavitusvalmidus

Roll D seob rollid A-C üheks pipeline'iks: `extract -> transform -> validate -> export`. Notebookis käivitame valideerimise ja piiratud ekspordi eraldi, et oleks selgelt näha, millised Week 8 põhinõuete failid tekivad.

In [ ]:
# Kontrollime, et põhitabelid ei oleks tühjad ja käibesummad klapiksid eri raportite vahel.
validate_results(results)

# Salvestame ainult Week 8 juhendi põhinõuete väljundid: CSV + weekly chart HTML + KPI summary HTML.
# Nii ei teki juurde graafikuid, mis on töö käigus eemaldatud.
output_dir = TEAM_DIR / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
date_str = datetime.now().strftime('%Y%m%d_%H%M%S')

# CSV väljundid: nädalased koondnäitajad ja KPI-d ajatempliga failinimega.
weekly_csv = output_dir / f'weekly_aggregates_{date_str}.csv'
kpis_csv = output_dir / f'kpis_{date_str}.csv'
results['weekly'].to_csv(weekly_csv, index=False, encoding='utf-8')
pd.DataFrame([results['kpis']]).to_csv(kpis_csv, index=False, encoding='utf-8')

# HTML väljundid: juhendis nõutud nädalane tulujoonis ja KPI kokkuvõte.
weekly_chart = output_dir / f'weekly_revenue_{date_str}.html'
kpi_chart = output_dir / f'kpi_summary_{date_str}.html'
create_weekly_chart(results['weekly']).write_html(weekly_chart)
create_kpi_summary(results['kpis']).write_html(kpi_chart)

paths = {
    'weekly_csv': weekly_csv,
    'kpis_csv': kpis_csv,
    'weekly_chart': weekly_chart,
    'kpi_chart': kpi_chart,
}

print('Valideerimine õnnestus.')
print('Väljundkaust:', output_dir)
print('Failide arv:', len(paths))

# Kuvame loodud failid, et oleks kohe näha, millised väljundid tekkisid.
for key, path in paths.items():
    print(f'{key}: {path}')

## Äritõlgendus Markole

Allolev tekst koostatakse pipeline'i RFM tulemuste põhjal automaatselt. See on mõeldud juhtimisotsuse toetamiseks: millised segmendid vajavad hoidmist, millised taasaktiveerimist ning kuidas kampaaniate mõju mõõta.

In [ ]:
# Prindime automaatselt koostatud ärilise tõlgenduse RFM tulemuste põhjal.
print(results['business_interpretation'])

# Teavituse kokkuvõte on sama info, mida pipeline saaks webhooki või emaili kaudu saata.
print('Teavituse lühikokkuvõte:')
display(pd.DataFrame([notification_summary(results)]))

## Kokkuvõte: soovitused ja refleksioon

**Mis Week 8-s muutus võrreldes Week 7-ga?**  
Week 7 RFM analüüs oli notebookipõhine analüüs. Week 8 muudab sama loogika tootmislikumaks pipeline'iks: andmed tulevad API kaudu, vead logitakse, päringuid proovitakse uuesti, tulemused valideeritakse ning väljundid salvestatakse automaatselt.

**Mida Marko saab kasutada?**  
Kõige praktilisemad väljundid selles notebookis on ajatempliga `weekly_aggregates_*.csv`, `kpis_*.csv`, `weekly_revenue_*.html` ja `kpi_summary_*.html`. Need vastavad Week 8 juhendi põhinõuetele: töödeldud tabelid ja jagatav HTML visualisatsioon.

**Soovitus järgmise sammuna**  
Käivitada At Risk segmendile kontrollgrupiga win-back kampaania ning mõõta 30 päeva jooksul taasostu määra, käivet kliendi kohta ja kampaania ROI-d.

**AI kasutamine**  
AI aitas Week 7 RFM loogika muuta Week 8 modulaarseks API pipeline'iks, lisada retry, fallback, logimise, valideerimise, ekspordi, dashboardi, marketingi tegevusplaani ja automaattestid.